# Phase 2: Paper Trading Demo

**Project**: PROTO:Market Maker - Paper Trading System  
**Phase**: 2 - Strategy & Execution  
**Date**: October 27, 2025

---

## Overview

This notebook demonstrates the **Phase 2** paper trading system, which transforms the backtesting algorithm into a fully event-driven trading platform.

### What You'll Learn

1. ✅ **Strategy Engine**: How the market-making strategy generates signals
2. ✅ **Mock Execution**: How orders are filled realistically
3. ✅ **Trading Session**: How all components work together
4. ✅ **Event Flow**: Complete end-to-end event processing
5. ✅ **Performance Analysis**: How to analyze trading results

### Architecture

```
Market Data → Strategy → Signal → OMS → Order → Execution → Fill → Portfolio
```

All components communicate via the **EventBus** with zero tight coupling.

## Setup

First, let's import all necessary modules and set up logging.

In [ ]:
import sys
import logging
from decimal import Decimal
from datetime import datetime, timedelta
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Import Phase 1 components
from core.event import EventBus, MarketDataEvent, SignalEvent
from core.enums import EventType
from engine.portfolio import PortfolioManager
from engine.oms import OrderManager
from engine.risk import RiskManager

# Import Phase 2 components
from engine.strategy import MarketMakerStrategy
from engine.execution import MockExecutionEngine
from paper_trading.session import TradingSession
from paper_trading.recorder import EventRecorder

print("✅ All imports successful!")

---

## Part 1: Strategy Engine Demo

The **MarketMakerStrategy** generates bid/ask signals based on:
- Current market price
- Inventory position
- Step parameter (2.9)

### Formula

```python
bid = (price - step) - step * max(inventory, 0) * 0.02
ask = (price + step) - step * min(inventory, 0) * 0.02
```

In [ ]:
# Create event bus and portfolio
bus = EventBus()
portfolio = PortfolioManager(bus, Decimal("500000"))

# Create strategy
strategy = MarketMakerStrategy(
    event_bus=bus,
    portfolio=portfolio,
    step=Decimal("2.9"),
    update_interval_seconds=15
)

print(f"Strategy initialized with step={strategy.step}")
print(f"Update interval: {strategy.update_interval.total_seconds()} seconds")

### Demo 1.1: Signal Generation with Zero Inventory

In [ ]:
# Capture signals
signals_received = []

def capture_signal(event):
    signals_received.append(event)

bus.subscribe(EventType.SIGNAL, capture_signal)

# Send market data (inventory = 0)
market_data = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250"),
    bid=Decimal("1249"),
    ask=Decimal("1251"),
    spread=Decimal("2")
)

bus.publish(market_data)
bus.process_events()

# Display signal
signal = signals_received[0]
print(f"\n📊 Signal Generated (Zero Inventory):")
print(f"  Contract: {signal.contract}")
print(f"  Bid Price: {signal.bid_price}")
print(f"  Ask Price: {signal.ask_price}")
print(f"  Reason: {signal.reason}")
print(f"  Spread: {signal.ask_price - signal.bid_price}")

### Demo 1.2: Inventory Effect on Bid/Ask

Let's see how inventory changes affect the bid/ask prices.

In [ ]:
# Test different inventory levels
inventories = [-3, -2, -1, 0, 1, 2, 3]
price = Decimal("1250")

print(f"\n📈 Inventory Effect on Bid/Ask (Price = {price}):\n")
print(f"{'Inventory':<12} {'Bid':<10} {'Ask':<10} {'Spread':<10} {'Direction':<15}")
print("-" * 60)

for inv in inventories:
    bid, ask = strategy.calculate_bid_ask(price, inv)
    spread = ask - bid
    
    if inv > 0:
        direction = "Long (want sell)"
    elif inv < 0:
        direction = "Short (want buy)"
    else:
        direction = "Flat (neutral)"
    
    print(f"{inv:<12} {bid:<10} {ask:<10} {spread:<10} {direction:<15}")

print("\n💡 Notice:")
print("  - Positive inventory → Bid decreases (want to sell)")
print("  - Negative inventory → Ask increases (want to buy)")
print("  - Zero inventory → Symmetric spread")

---

## Part 2: Mock Execution Engine Demo

The **MockExecutionEngine** simulates realistic order execution:
- BID orders fill when market price ≤ bid price
- ASK orders fill when market price ≥ ask price
- Applies realistic fees (20 VND per contract)

In [ ]:
# Create fresh components
bus2 = EventBus()
portfolio2 = PortfolioManager(bus2, Decimal("500000"))
risk2 = RiskManager(portfolio2)
oms2 = OrderManager(bus2, risk2)
strategy2 = MarketMakerStrategy(bus2, portfolio2, Decimal("2.9"))
execution2 = MockExecutionEngine(bus2)

# Capture fills
fills_received = []

def capture_fill(event):
    fills_received.append(event)

bus2.subscribe(EventType.FILL, capture_fill)

print("✅ Execution engine ready")
print(f"   Fee per contract: {MockExecutionEngine.FEE_PER_CONTRACT} VND")

### Demo 2.1: Complete Trading Cycle

In [ ]:
print("\n🔄 Trading Cycle Demo:\n")

# Step 1: Initial market data → generates signal
print("Step 1: Market data at 1250 → Strategy generates signal")
market_data1 = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250"),
    bid=Decimal("1249"),
    ask=Decimal("1251"),
    spread=Decimal("2")
)
bus2.publish(market_data1)
bus2.process_events()
print(f"  ✓ Signal generated: BID @ 1247.1, ASK @ 1252.9")

# Step 2: Process signal → creates orders
print("\nStep 2: OMS receives signal → Creates and submits orders")
bus2.process_events()
print(f"  ✓ Orders submitted: {len(oms2.active_orders)} active orders")

# Step 3: Price drops → fills bid order
print("\nStep 3: Market price drops to 1247.0 → Fills BID order")
market_data2 = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1247.0"),
    bid=Decimal("1246"),
    ask=Decimal("1248"),
    spread=Decimal("2")
)
bus2.publish(market_data2)
bus2.process_events()

if fills_received:
    fill = fills_received[0]
    print(f"  ✓ Order filled!")
    print(f"    - Side: {fill.side}")
    print(f"    - Price: {fill.fill_price}")
    print(f"    - Quantity: {fill.fill_quantity}")
    print(f"    - Fee: {fill.fee} VND")

# Step 4: Portfolio updates
print("\nStep 4: Portfolio updates position")
bus2.process_events()
position = portfolio2.get_position("VN30F1M")
print(f"  ✓ Position updated:")
print(f"    - Quantity: {position.quantity}")
print(f"    - Average Price: {position.average_price}")
print(f"    - Cash: {portfolio2.cash}")

# Step 5: Strategy generates new signal
print("\nStep 5: Strategy generates new signal (inventory changed)")
print(f"  ✓ New orders will reflect inventory = {position.quantity}")

---

## Part 3: Trading Session Demo

The **TradingSession** orchestrates all components to run a complete backtest.

### Demo 3.1: Create Sample Data

In [ ]:
# Create realistic sample data
timestamps = pd.date_range(start='2025-01-01 09:00:00', periods=20, freq='15S')

# Simulate price movements
prices = [
    1250, 1251, 1250, 1249, 1248,  # Drop to hit bid
    1249, 1250, 1251, 1252, 1253,  # Rise to hit ask
    1252, 1251, 1250, 1249, 1250,  # More movements
    1251, 1252, 1253, 1254, 1255   # Uptrend
]

data = pd.DataFrame({
    'datetime': timestamps,
    'tickersymbol': ['VN30F1M'] * 20,
    'price': prices,
    'best-bid': [p - 1 for p in prices],
    'best-ask': [p + 1 for p in prices],
    'spread': [2] * 20
})

print("📊 Sample Data Created:")
print(data.head(10))
print(f"\nTotal data points: {len(data)}")
print(f"Price range: {min(prices)} - {max(prices)}")

### Demo 3.2: Run Trading Session

In [ ]:
# Create trading session
session = TradingSession(
    initial_capital=Decimal("500000"),
    step=Decimal("2.9"),
    update_interval_seconds=15
)

print("🚀 Running trading session...\n")

# Run backtest
summary = session.run_backtest(data)

print("\n✅ Session complete!")

### Demo 3.3: Analyze Results

In [ ]:
print("\n" + "="*70)
print("TRADING SESSION RESULTS")
print("="*70)

# Portfolio summary
portfolio_summary = summary['portfolio']
print("\n📊 Portfolio Summary:")
print(f"  Initial Capital: {portfolio_summary['initial_capital']:>15,.2f} VND")
print(f"  Final NAV:       {portfolio_summary['final_nav']:>15,.2f} VND")
print(f"  Cash:            {portfolio_summary['cash']:>15,.2f} VND")
print(f"  Total Return:    {portfolio_summary['total_return']:>15,.2f}%")

# Positions
if portfolio_summary['positions']:
    print("\n📈 Open Positions:")
    for pos in portfolio_summary['positions']:
        if pos['quantity'] != 0:
            print(f"  {pos['contract']:>10}: qty={pos['quantity']:>3}, "
                  f"avg_px={pos['average_price']:>8.1f}, "
                  f"pnl={pos['total_pnl']:>10.2f}")
else:
    print("\n📈 Positions: Flat (all positions closed)")

# Orders
orders = summary['orders']
print("\n📋 Order Statistics:")
print(f"  Total Orders:     {orders['total_orders']:>10}")
print(f"  Filled Orders:    {orders['filled_orders']:>10}")
print(f"  Cancelled Orders: {orders['cancelled_orders']:>10}")
print(f"  Active Orders:    {orders['active_orders']:>10}")

# Calculate fill rate
if orders['total_orders'] > 0:
    fill_rate = (orders['filled_orders'] / orders['total_orders']) * 100
    print(f"\n  Fill Rate:        {fill_rate:>10.1f}%")

print("\n" + "="*70)

---

## Part 4: Event Flow Visualization

Let's trace the complete event flow for a single market data update.

In [ ]:
# Create fresh session with event tracking
bus3 = EventBus()
portfolio3 = PortfolioManager(bus3, Decimal("500000"))
risk3 = RiskManager(portfolio3)
oms3 = OrderManager(bus3, risk3)
strategy3 = MarketMakerStrategy(bus3, portfolio3, Decimal("2.9"))
execution3 = MockExecutionEngine(bus3)

# Track all events
event_log = []

def log_event(event_type):
    def wrapper(event):
        event_log.append({
            'type': event_type,
            'timestamp': event.timestamp,
            'data': str(event)[:80]  # Truncate for readability
        })
    return wrapper

bus3.subscribe(EventType.MARKET_DATA, log_event('MARKET_DATA'))
bus3.subscribe(EventType.SIGNAL, log_event('SIGNAL'))
bus3.subscribe(EventType.ORDER, log_event('ORDER'))
bus3.subscribe(EventType.FILL, log_event('FILL'))

print("\n🔍 Event Flow Tracing:\n")
print("Sending market data and processing events...\n")

# Send market data
market_data = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250"),
    bid=Decimal("1249"),
    ask=Decimal("1251"),
    spread=Decimal("2")
)
bus3.publish(market_data)
bus3.process_events()

# Process signals
bus3.process_events()

# Send price that fills bid
market_data2 = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1247.0"),
    bid=Decimal("1246"),
    ask=Decimal("1248"),
    spread=Decimal("2")
)
bus3.publish(market_data2)
bus3.process_events()

# Display event flow
print("Event Sequence:")
print("-" * 80)
for i, evt in enumerate(event_log, 1):
    print(f"{i:>2}. [{evt['type']:<12}] {evt['timestamp'].strftime('%H:%M:%S.%f')[:-3]}")

print("\n💡 Event Flow Summary:")
print(f"  Total events: {len(event_log)}")
print(f"  Market Data events: {sum(1 for e in event_log if e['type'] == 'MARKET_DATA')}")
print(f"  Signal events: {sum(1 for e in event_log if e['type'] == 'SIGNAL')}")
print(f"  Order events: {sum(1 for e in event_log if e['type'] == 'ORDER')}")
print(f"  Fill events: {sum(1 for e in event_log if e['type'] == 'FILL')}")

---

## Part 5: Event Recording Demo

The **EventRecorder** allows you to record all events for later analysis.

In [ ]:
import tempfile
from pathlib import Path

# Create temporary file for recording
temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.jsonl')
temp_path = temp_file.name
temp_file.close()

print(f"\n📹 Recording events to: {temp_path}\n")

# Create session with recorder
bus4 = EventBus()
portfolio4 = PortfolioManager(bus4, Decimal("500000"))
risk4 = RiskManager(portfolio4)
oms4 = OrderManager(bus4, risk4)
strategy4 = MarketMakerStrategy(bus4, portfolio4, Decimal("2.9"))
execution4 = MockExecutionEngine(bus4)

# Create recorder
with EventRecorder(temp_path) as recorder:
    # Subscribe recorder to all events
    for event_type in [EventType.MARKET_DATA, EventType.SIGNAL, EventType.ORDER, EventType.FILL]:
        bus4.subscribe(event_type, recorder.record)
    
    # Send some market data
    for i in range(3):
        market_data = MarketDataEvent(
            timestamp=datetime.now(),
            contract="VN30F1M",
            price=Decimal(str(1250 + i)),
            bid=Decimal(str(1249 + i)),
            ask=Decimal(str(1251 + i)),
            spread=Decimal("2")
        )
        bus4.publish(market_data)
        bus4.process_events()
        bus4.process_events()  # Process signals
    
    print(f"Recorded {recorder.event_count} events")

# Read back the recorded events
print(f"\n📖 Reading back events...\n")

from paper_trading.recorder import EventReplayer

replayer = EventReplayer(temp_path)
stats = replayer.get_statistics()

print("Event Statistics:")
print(f"  Total events: {stats['total_events']}")
for event_type, count in stats['by_type'].items():
    print(f"  {event_type}: {count}")

# Show first few events
print("\nFirst 3 recorded events:")
for i, event_data in enumerate(replayer.replay(), 1):
    if i > 3:
        break
    print(f"\n{i}. {event_data['event_type']} @ {event_data['timestamp']}")
    if 'contract' in event_data['data']:
        print(f"   Contract: {event_data['data']['contract']}")

# Cleanup
Path(temp_path).unlink()
print(f"\n✅ Demo complete. Temporary file cleaned up.")

---

## Part 6: Performance Comparison

Let's compare different step parameters to see their effect on performance.

In [ ]:
# Create test data
test_data = pd.DataFrame({
    'datetime': pd.date_range(start='2025-01-01 09:00:00', periods=50, freq='15S'),
    'tickersymbol': ['VN30F1M'] * 50,
    'price': [1250 + (i % 10 - 5) for i in range(50)],  # Oscillating price
    'best-bid': [1248 + (i % 10 - 5) for i in range(50)],
    'best-ask': [1252 + (i % 10 - 5) for i in range(50)],
    'spread': [2] * 50
})

# Test different step values
step_values = [Decimal("1.5"), Decimal("2.9"), Decimal("4.0")]
results = []

print("\n🔬 Testing different step parameters...\n")

for step in step_values:
    session = TradingSession(
        initial_capital=Decimal("500000"),
        step=step,
        update_interval_seconds=15
    )
    
    summary = session.run_backtest(test_data)
    
    results.append({
        'step': float(step),
        'final_nav': summary['portfolio']['final_nav'],
        'return': summary['portfolio']['total_return'],
        'filled_orders': summary['orders']['filled_orders'],
        'total_orders': summary['orders']['total_orders']
    })

# Display comparison
print("\n📊 Step Parameter Comparison:\n")
print(f"{'Step':<8} {'Final NAV':<15} {'Return %':<12} {'Fills':<10} {'Fill Rate %':<12}")
print("-" * 70)

for r in results:
    fill_rate = (r['filled_orders'] / r['total_orders'] * 100) if r['total_orders'] > 0 else 0
    print(f"{r['step']:<8.1f} {r['final_nav']:>14,.2f} {r['return']:>11.2f} "
          f"{r['filled_orders']:>9} {fill_rate:>11.1f}")

print("\n💡 Insights:")
print("  - Smaller step = tighter spread = more fills but less profit per trade")
print("  - Larger step = wider spread = fewer fills but more profit per trade")
print("  - Optimal step (2.9) balances fill rate and profitability")

---

## Summary

### What We Learned

1. ✅ **Strategy Engine**: Generates signals based on inventory and time/events
2. ✅ **Mock Execution**: Realistically simulates order fills with proper fees
3. ✅ **Trading Session**: Orchestrates all components seamlessly
4. ✅ **Event Flow**: Complete event-driven architecture with EventBus
5. ✅ **Event Recording**: Capture and replay events for analysis
6. ✅ **Performance**: Different parameters affect results differently

### Key Takeaways

- **Event-Driven Design**: All components communicate via events, enabling:
  - Loose coupling
  - Easy testing
  - Real-time and historical data support

- **Inventory Management**: The strategy automatically adjusts bid/ask based on position

- **Realistic Execution**: Orders fill only when price crosses levels, with proper fees

- **Complete Integration**: All Phase 1 + Phase 2 components work together seamlessly

### Next Steps

1. **Try with Real Data**: Load historical CSV data and run full backtests
2. **Optimize Parameters**: Test different step values and update intervals
3. **Phase 3**: Add Redis Pub/Sub for real-time streaming data
4. **Production**: Deploy for paper trading with live market data

---

## Additional Resources

- **Phase 2 Specification**: `internal-docs/paper-trading-phase-2-spec.md`
- **Completion Report**: `markdown-docs/phase-2-completion-report.md`
- **Phase 1 Tutorial**: `examples/core-engine-architecture-guide.ipynb`
- **CLI Usage**: `python -m paper_trading.main --help`

---

**Happy Paper Trading!** 🚀